In [ ]:
"""
CSV File Handler for Processing Covid-19 Data.

This script downloads daily Covid-19 reports from GitHub, processes them,
and generates visualizations including time-series plots, bar charts,
and correlation matrices.

Author: Vinit Shah
Date: 25/02/2025

https://github.com/vinitrshah03/OST
"""

import csv
import os
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from jobspy import scrape_jobs
from wordcloud import WordCloud

In [2]:

# Configure Logging
log_file = "C:/Users/Rithin/OneDrive/Desktop/log.txt"
# Define file paths
output_dir = os.path.expanduser("C:/Users/Rithin/OneDrive/Desktop")
csv_path = os.path.join(output_dir, "jobs.csv")
json_path = os.path.join(output_dir, "jobs.json")

In [3]:
logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
console_handler.setFormatter(formatter)
logging.getLogger().addHandler(console_handler)

logging.info("Script started.")


2025-02-27 10:45:38,572 - INFO - Script started.


In [4]:
# Collect user input
try:
    search_term = input("Enter the job you are looking for: ")
    location = input("Enter location: ")
    country_indeed = input("Enter country: ")
    job_type = input("Enter job type (fulltime/parttime/internship/contract): ")
    logging.info(f"User Input - Job: {search_term}, Location: {location}, Country: {country_indeed}, Job Type: {job_type}")
except Exception as e:
    logging.error(f"Error collecting user input: {e}")
    exit(1)


2025-02-27 10:46:03,315 - INFO - User Input - Job: Software Engineer, Location: Pune, Country: India, Job Type: fulltime


In [5]:
# Scrape jobs
try:
    job_data = scrape_jobs(
        site_name=["indeed", "linkedin", "glassdoor", "google", "bayt"],
        search_term=search_term,
        location=location,
        country_indeed=country_indeed,
        linkedin_fetch_description=True,
        job_type=job_type,
    )
    logging.info(f"Successfully scraped {len(job_data)} job postings.")
except Exception as e:
    logging.error(f"Error scraping job data: {e}")
    exit(1)

2025-02-27 10:46:33,350 - INFO - JobSpy:Linkedin - finished scraping
2025-02-27 10:46:33,464 - INFO - Successfully scraped 75 job postings.


In [6]:
# Save to CSV
try:
    job_data.to_csv(csv_path, quoting=csv.QUOTE_NONNUMERIC, escapechar="\\", index=False)
    logging.info(f"CSV file saved: {csv_path}")
except Exception as e:
    logging.error(f"Error saving CSV file: {e}")

# Save to JSON
try:
    job_data.to_json(json_path, orient="records", indent=4)
    logging.info(f"JSON file saved: {json_path}")
except Exception as e:
    logging.error(f"Error saving JSON file: {e}")

print(f"Job searches saved to:\n- {csv_path}\n- {json_path}")

2025-02-27 10:46:37,996 - INFO - CSV file saved: C:/Users/Rithin/OneDrive/Desktop\jobs.csv
2025-02-27 10:46:38,003 - INFO - JSON file saved: C:/Users/Rithin/OneDrive/Desktop\jobs.json


Job searches saved to:
- C:/Users/Rithin/OneDrive/Desktop\jobs.csv
- C:/Users/Rithin/OneDrive/Desktop\jobs.json


In [7]:
""" Data Analysis and Visualization """

# Load CSV file
try:
    df = pd.read_csv(csv_path)
    logging.info("CSV file loaded successfully.")
except Exception as e:
    logging.error(f"Error loading CSV file: {e}")
    exit(1)

# Convert date_posted to datetime
df["date_posted"] = pd.to_datetime(df["date_posted"], errors="coerce")

# Convert salary fields to numeric
df["min_amount"] = pd.to_numeric(df["min_amount"], errors="coerce")
df["max_amount"] = pd.to_numeric(df["max_amount"], errors="coerce")

2025-02-27 10:46:42,003 - INFO - CSV file loaded successfully.


In [8]:
# Create a directory for saving graphs
graph_dir = os.path.join(output_dir, "Job Scraping Charts")
os.makedirs(graph_dir, exist_ok=True)

def save_plot(plt_obj, filename):
    try:
        plt_obj.savefig(os.path.join(graph_dir, filename))
        logging.info(f"Graph saved: {filename}")
    except Exception as e:
        logging.error(f"Error saving graph {filename}: {e}")
    finally:
        plt_obj.close()

In [9]:
### 1. Bar Chart - Job Postings by Site
try:
    plt.figure(figsize=(10, 5))
    sns.countplot(data=df, x="site", order=df["site"].value_counts().index, palette="viridis")
    plt.xticks(rotation=45)
    plt.title("Job Postings by Site")
    plt.xlabel("Job Site")
    plt.ylabel("Number of Job Postings")
    save_plot(plt, "job_postings_by_site.png")
except Exception as e:
    logging.error(f"Error creating bar chart: {e}")

C:\Users\Rithin\AppData\Local\Temp\ipykernel_25768\1630124234.py:4: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df, x="site", order=df["site"].value_counts().index, palette="viridis")
2025-02-27 10:46:50,173 - INFO - Graph saved: job_postings_by_site.png


In [10]:
### 2. Pie Chart - Job Type Distribution
try:
    plt.figure(figsize=(8, 8))
    df["job_type"].value_counts().plot.pie(autopct="%1.1f%%", cmap="coolwarm", startangle=90)
    plt.title("Job Type Distribution")
    save_plot(plt, "job_type_distribution.png")
except Exception as e:
    logging.error(f"Error creating pie chart: {e}")

2025-02-27 10:46:54,330 - INFO - Graph saved: job_type_distribution.png


In [11]:
### 3. Line Graph - Job Postings Over Time
try:
    plt.figure(figsize=(12, 6))
    df.groupby(df["date_posted"].dt.date)["id"].count().plot(marker="o", linestyle="-")
    plt.xticks(rotation=45)
    plt.title("Job Postings Over Time")
    plt.xlabel("Date")
    plt.ylabel("Number of Job Postings")
    plt.grid(True)
    save_plot(plt, "job_postings_over_time.png")
except Exception as e:
    logging.error(f"Error creating line graph: {e}")

2025-02-27 10:46:57,906 - INFO - Graph saved: job_postings_over_time.png


In [12]:
### 4. Heatmap - Correlation Between Salary Fields
try:
    plt.figure(figsize=(8, 6))
    sns.heatmap(df[["min_amount", "max_amount"]].corr(), annot=True, cmap="coolwarm", linewidths=1)
    plt.title("Correlation Between Salary Fields")
    save_plot(plt, "salary_correlation_heatmap.png")
except Exception as e:
    logging.error(f"Error creating heatmap: {e}")

2025-02-27 10:47:00,984 - INFO - Graph saved: salary_correlation_heatmap.png


In [13]:
### 5. Word Cloud - Common Skills in Job Descriptions
try:
    text = " ".join(df["description"].dropna())  # Combine all descriptions
    wordcloud = WordCloud(width=800, height=400, background_color="white").generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title("Commonly Mentioned Skills in Job Descriptions")
    save_plot(plt, "job_description_wordcloud.png")
except Exception as e:
    logging.error(f"Error creating word cloud: {e}")

2025-02-27 10:47:03,978 - INFO - Graph saved: job_description_wordcloud.png


In [14]:
logging.info("Graphs saved successfully.")

print(f"Graphs saved successfully in the '{graph_dir}' folder.")
print("Logs stored in log.txt")

2025-02-27 10:47:05,812 - INFO - Graphs saved successfully.


Graphs saved successfully in the 'C:/Users/Rithin/OneDrive/Desktop\Job Scraping Charts' folder.
Logs stored in log.txt
